In [1]:
# Loading the dataset in - since it is so large, it must be done using batch processing

import pandas as pd
import numpy as np
import kagglehub


csv_path = kagglehub.dataset_download(
    "sobhanmoosavi/us-accidents",
    path="US_Accidents_March23.csv",
)

batch_size = 5000
batches = []
batch_num = 0

# To reduce the memory usage, we will do the filtering of the cities when processing
# We chose these cities because they are the cities with most data in dataset, and also are distinct and large population centers
cities = ["Miami", "Houston", "Los Angeles"]
filtered_batches = []

for batch in pd.read_csv(csv_path, chunksize=50000):
    filtered = batch[batch["City"].isin(cities)]
    
    if not filtered.empty:
        filtered_batches.append(filtered)

df_trimmed = pd.concat(filtered_batches, ignore_index=True)

print(df_trimmed.shape)

/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(513017, 46)


In [2]:
# select columns to be dropped
columns_dropped = [
    # zero/near-zero variance
    'Turning_Loop',
    'Roundabout',
    'No_Exit',
    'Traffic_Calming',
    'Bump',

    # redundant or irrelevant
    'Country',
    'State',
    'End_Lat',
    'End_Lng',
    'Airport_Code',
    'Weather_Timestamp',
    'County',
    'Timezone',
    'Civil_Twilight',
    'Nautical_Twilight',
    'Astronomical_Twilight',
    'Wind_Chill(F)',
    'ID',
    'Description',
    'Street',
]

# # create a copy to keep a primary df and apply transformation
d2 = df_trimmed.copy().drop(columns_dropped, axis=1)

print(f"Columns before: {df_trimmed.shape[1]}")
print(f"Columns after: {d2.shape[1]}")
print(f"Remaining columns: {list(d2.columns)}")

Columns before: 46
Columns after: 26
Remaining columns: ['Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'Distance(mi)', 'City', 'Zipcode', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Crossing', 'Give_Way', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset']


In [3]:
# checking for null values
d2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 513017 entries, 0 to 513016
Data columns (total 26 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Source             513017 non-null  object 
 1   Severity           513017 non-null  int64  
 2   Start_Time         513017 non-null  object 
 3   End_Time           513017 non-null  object 
 4   Start_Lat          513017 non-null  float64
 5   Start_Lng          513017 non-null  float64
 6   Distance(mi)       513017 non-null  float64
 7   City               513017 non-null  object 
 8   Zipcode            513017 non-null  object 
 9   Temperature(F)     507387 non-null  float64
 10  Humidity(%)        507117 non-null  float64
 11  Pressure(in)       509236 non-null  float64
 12  Visibility(mi)     508298 non-null  float64
 13  Wind_Direction     506361 non-null  object 
 14  Wind_Speed(mph)    471323 non-null  float64
 15  Precipitation(in)  371056 non-null  float64
 16  We

## Null Value Handling

In [4]:
# populating null values instead of completely dropping using forward fill (time-series) with a 3 hour gap for similar results (can't use KNN because data too large)

import numpy as np

# sort first
d2 = d2.sort_values(['City', 'Start_Time'])

# change to datetime (valuerror showed to add in format='ISO8601' as a parameter)
d2['Start_Time'] = pd.to_datetime(d2['Start_Time'], format='ISO8601')
d2['End_Time'] = pd.to_datetime(d2['End_Time'], format='ISO8601')

# select weather columns
weather_cols = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 
                'Visibility(mi)', 'Wind_Direction', 
                'Wind_Speed(mph)', 'Weather_Condition']

# tweak this threshold as needed, currently 3 hours
max_gap = pd.Timedelta(hours=3)

# forward fill weather values, but only if the time gap between the current row and the last known value is within the max_gap threshold
def ffill_with_time_limit(group, cols, max_gap):
    for col in cols:
        filled = group[col].ffill()

        # calculate time diff between each row and the last non-null row
        last_valid_time = group['Start_Time'].where(group[col].notna()).ffill()
        time_gap = group['Start_Time'] - last_valid_time

        # only accept the fill if the gap is within threshold
        group[col] = filled.where(time_gap <= max_gap, other=np.nan)
    return group

# apply the function to each city group (done by city to avoid filling across different cities which may have different weather conditions)
city_groups = []
for city, group in d2.groupby('City'):
    filled_group = ffill_with_time_limit(group, weather_cols, max_gap)
    city_groups.append(filled_group)

# concatenate the groups back together
d2 = pd.concat(city_groups).sort_values(['City', 'Start_Time'])

# precipitation still just fills with 0
d2['Precipitation(in)'] = d2['Precipitation(in)'].fillna(0)

# check remaining nulls
print(d2.isnull().sum())

Source                  0
Severity                0
Start_Time              0
End_Time                0
Start_Lat               0
Start_Lng               0
Distance(mi)            0
City                    0
Zipcode                 0
Temperature(F)        625
Humidity(%)           634
Pressure(in)          572
Visibility(mi)        595
Wind_Direction        611
Wind_Speed(mph)      9566
Precipitation(in)       0
Weather_Condition     586
Amenity                 0
Crossing                0
Give_Way                0
Junction                0
Railway                 0
Station                 0
Stop                    0
Traffic_Signal          0
Sunrise_Sunset          6
dtype: int64


In [5]:
# drop the remaining null columns
d2 = d2.dropna()
# reset the index for simplicity
d2.reset_index(drop=True)

,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),City,Zipcode,Temperature(F),...,Weather_Condition,Amenity,Crossing,Give_Way,Junction,Railway,Station,Stop,Traffic_Signal,Sunrise_Sunset
0,Source2,2,2016-06-14 20:21:49,2016-06-14 21:07:49,29.757492,-95.365791,0.000,Houston,77002-6308,86.0,...,Clear,False,False,False,False,False,True,False,True,Day
1,Source2,2,2016-06-14 20:26:55,2016-06-14 21:12:35,29.821486,-95.368080,0.000,Houston,77022-4842,84.2,...,Clear,False,False,False,False,False,False,True,False,Night
2,Source1,2,2016-06-17 15:44:51,2016-06-17 21:44:51,29.784610,-95.617360,0.940,Houston,77079,93.0,...,Partly Cloudy,False,False,True,True,False,False,False,False,Day
3,Source1,3,2016-06-17 15:53:54,2016-06-17 21:53:54,29.784750,-95.683090,1.959,Houston,77094,93.0,...,Partly Cloudy,False,False,False,False,False,False,False,False,Day
4,Source2,2,2016-06-21 09:27:32,2016-06-21 10:42:32,29.859404,-95.356445,0.000,Houston,77093-4446,84.9,...,Scattered Clouds,False,True,False,False,False,False,True,False,Day
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
503343,Source1,2,2023-03-20 07:06:30,2023-03-20 09:29:00,25.811510,-80.199829,3.469,Miami,33127,60.0,...,Cloudy,False,False,False,True,False,False,False,False,Night
503344,Source1,2,2023-03-26 16:11:30,2023-03-26 17:01:13,25.787840,-80.209907,0.386,Miami,33136,86.0,...,Mostly Cloudy,False,False,False,False,False,False,False,False,Day
503345,Source1,2,2023-03-26 16:29:39,2023-03-26 21:29:38,25.787316,-80.210999,0.800,Miami,33136,84.0,...,Partly Cloudy,False,False,False,False,False,False,False,False,Day
503346,Source1,2,2023-03-30 06:41:00,2023-03-30 09:24:30,25.897225,-80.380845,0.701,Miami,33178,73.0,...,Partly Cloudy,False,False,False,False,False,False,False,False,Night


## Feature Engineering

In [6]:
# Extracting more relevant predictors from the start time, such as hour, day of week, weekend, etc
d2['Start_Date'] = d2['Start_Time'].dt.date
d2['Hour'] = d2['Start_Time'].dt.hour
d2["Day_of_Week"] = d2['Start_Time'].dt.dayofweek
d2['Month'] = d2['Start_Time'].dt.month
d2['Weekend'] = d2['Start_Time'].dt.weekday >= 5

In [7]:
import holidays

# create a US holidays object
us_holidays = holidays.UnitedStates()
# create a Holiday column that indicates whether the date is a holiday
d2['Holiday'] = d2['Start_Date'].apply(lambda x: x in us_holidays).astype(int)
d2['Holiday_Name'] = d2['Start_Time'].dt.date.apply(lambda x: us_holidays.get(x))
d2["Holiday_Name"].fillna("Not Holiday", inplace=True)

/var/folders/x7/sl4zn7qx64s6wq122_6jw9_00000gn/T/ipykernel_99917/1919409221.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  d2["Holiday_Name"].fillna("Not Holiday", inplace=True)


In [8]:
d2[d2['Holiday'] == 1]

,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),City,Zipcode,Temperature(F),...,Stop,Traffic_Signal,Sunrise_Sunset,Start_Date,Hour,Day_of_Week,Month,Weekend,Holiday,Holiday_Name
212767,Source1,3,2016-07-04 00:17:22,2016-07-04 06:17:22,29.777540,-95.371850,0.932,Houston,77007,82.4,...,False,False,Night,2016-07-04,0,0,7,False,1,Independence Day
212768,Source1,4,2016-07-04 02:02:40,2016-07-04 08:02:40,29.693499,-95.525209,2.248,Houston,77074,82.4,...,False,False,Night,2016-07-04,2,0,7,False,1,Independence Day
212769,Source1,3,2016-07-04 04:46:08,2016-07-04 10:46:08,29.691730,-95.283620,0.847,Houston,77017,81.0,...,False,False,Night,2016-07-04,4,0,7,False,1,Independence Day
30226,Source2,2,2016-07-04 06:00:56,2016-07-04 06:47:55,29.715075,-95.571510,0.000,Houston,77072,80.1,...,False,False,Night,2016-07-04,6,0,7,False,1,Independence Day
30227,Source2,2,2016-07-04 06:36:44,2016-07-04 07:06:44,29.737444,-95.510094,0.000,Houston,77063-2901,80.6,...,False,True,Day,2016-07-04,6,0,7,False,1,Independence Day
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279668,Source1,2,2023-01-16 21:01:30,2023-01-16 21:31:00,25.833118,-80.241320,0.131,Miami,33147-7799,60.0,...,False,False,Night,2023-01-16,21,0,1,False,1,Martin Luther King Jr. Day
294840,Source1,2,2023-01-16 21:14:44,2023-01-16 22:32:02,25.807459,-80.292632,0.018,Miami,33166,63.0,...,False,False,Night,2023-01-16,21,0,1,False,1,Martin Luther King Jr. Day
342717,Source1,2,2023-01-16 21:29:57,2023-01-17 01:56:06,25.759270,-80.265129,0.128,Miami,33134-2725,61.0,...,False,False,Night,2023-01-16,21,0,1,False,1,Martin Luther King Jr. Day
328231,Source1,2,2023-01-16 22:17:13,2023-01-16 23:35:03,25.699823,-80.462600,0.092,Miami,33193,55.0,...,False,False,Night,2023-01-16,22,0,1,False,1,Martin Luther King Jr. Day


### Converting Data to Numerical

In [ ]:
#pd.set_option('display.max_columns', None)

In [10]:
# One Hot Encoding the cities column
cities = pd.get_dummies(d2['City'], prefix='City')
d2 = pd.concat([d2, cities], axis=1)
d2

,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),City,Zipcode,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Crossing,Give_Way,Junction,Railway,Station,Stop,Traffic_Signal,Sunrise_Sunset,Start_Date,Hour,Day_of_Week,Month,Weekend,Holiday,Holiday_Name,City_Houston,City_Los Angeles,City_Miami
29687,Source2,2,2016-06-14 20:21:49,2016-06-14 21:07:49,29.757492,-95.365791,0.000,Houston,77002-6308,86.0,66.0,29.84,8.0,ESE,9.2,0.0,Clear,False,False,False,False,False,True,False,True,Day,2016-06-14,20,1,6,False,0,Not Holiday,True,False,False
29686,Source2,2,2016-06-14 20:26:55,2016-06-14 21:12:35,29.821486,-95.368080,0.000,Houston,77022-4842,84.2,70.0,29.84,8.0,ESE,9.2,0.0,Clear,False,False,False,False,False,False,True,False,Night,2016-06-14,20,1,6,False,0,Not Holiday,True,False,False
212552,Source1,2,2016-06-17 15:44:51,2016-06-17 21:44:51,29.784610,-95.617360,0.940,Houston,77079,93.0,50.0,29.94,10.0,SSE,10.4,0.0,Partly Cloudy,False,False,True,True,False,False,False,False,Day,2016-06-17,15,4,6,False,0,Not Holiday,True,False,False
212551,Source1,3,2016-06-17 15:53:54,2016-06-17 21:53:54,29.784750,-95.683090,1.959,Houston,77094,93.0,50.0,29.94,10.0,SSE,10.4,0.0,Partly Cloudy,False,False,False,False,False,False,False,False,Day,2016-06-17,15,4,6,False,0,Not Holiday,True,False,False
29692,Source2,2,2016-06-21 09:27:32,2016-06-21 10:42:32,29.859404,-95.356445,0.000,Houston,77093-4446,84.9,69.0,30.14,10.0,WNW,4.6,0.0,Scattered Clouds,False,True,False,False,False,False,True,False,Day,2016-06-21,9,1,6,False,0,Not Holiday,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224286,Source1,2,2023-03-20 07:06:30,2023-03-20 09:29:00,25.811510,-80.199829,3.469,Miami,33127,60.0,96.0,30.08,10.0,NW,9.0,0.0,Cloudy,False,False,False,True,False,False,False,False,Night,2023-03-20,7,0,3,False,0,Not Holiday,False,False,True
224198,Source1,2,2023-03-26 16:11:30,2023-03-26 17:01:13,25.787840,-80.209907,0.386,Miami,33136,86.0,57.0,30.04,10.0,SSE,8.0,0.0,Mostly Cloudy,False,False,False,False,False,False,False,False,Day,2023-03-26,16,6,3,True,0,Not Holiday,False,False,True
223946,Source1,2,2023-03-26 16:29:39,2023-03-26 21:29:38,25.787316,-80.210999,0.800,Miami,33136,84.0,61.0,30.03,10.0,SE,12.0,0.0,Partly Cloudy,False,False,False,False,False,False,False,False,Day,2023-03-26,16,6,3,True,0,Not Holiday,False,False,True
224753,Source1,2,2023-03-30 06:41:00,2023-03-30 09:24:30,25.897225,-80.380845,0.701,Miami,33178,73.0,90.0,30.07,10.0,N,3.0,0.0,Partly Cloudy,False,False,False,False,False,False,False,False,Night,2023-03-30,6,3,3,False,0,Not Holiday,False,False,True


In [11]:
# Since there is too many weather conditions, must consolidate them before doing one hot encoding

weather_conditions_map = {
    'Fair': 'Clear',
    'Clear': 'Clear',
    'Fair / Windy': 'Clear',

    'Mostly Cloudy': 'Cloudy',
    'Partly Cloudy': 'Cloudy',
    'Cloudy': 'Cloudy',
    'Overcast': 'Cloudy',
    'Scattered Clouds': 'Cloudy',
    'Mostly Cloudy / Windy': 'Cloudy',
    'Cloudy / Windy': 'Cloudy',
    'Partly Cloudy / Windy': 'Cloudy',

    'Light Rain': 'Rain/Drizzle',
    'Rain': 'Rain/Drizzle',
    'Heavy Rain': 'Rain/Drizzle',
    'Light Rain / Windy': 'Rain/Drizzle',
    'Light Drizzle': 'Rain/Drizzle',
    'Drizzle': 'Rain/Drizzle',
    'Rain / Windy': 'Rain/Drizzle',
    'Heavy Rain / Windy': 'Rain/Drizzle',
    'Drizzle and Fog': 'Rain/Drizzle',
    'Showers in the Vicinity': 'Rain/Drizzle',
    'Light Rain Showers': 'Rain/Drizzle',
    'Rain Showers': 'Rain/Drizzle',

    'Thunder in the Vicinity': 'Stormy',
    'Thunder': 'Stormy',
    'T-Storm': 'Stormy',
    'Heavy T-Storm': 'Stormy',
    'Thunderstorm': 'Stormy',
    'Light Rain with Thunder': 'Stormy',
    'Light Thunderstorms and Rain': 'Stormy',
    'Heavy Thunderstorms and Rain': 'Stormy',
    'Thunderstorms and Rain': 'Stormy',
    'Heavy T-Storm / Windy': 'Stormy',
    'T-Storm / Windy': 'Stormy',
    'Thunder / Windy': 'Stormy',
    'Squalls / Windy': 'Stormy',
    'Funnel Cloud': 'Stormy',

    'Haze': 'Visibility Issues',
    'Fog': 'Visibility Issues',
    'Shallow Fog': 'Visibility Issues',
    'Mist': 'Visibility Issues',
    'Patches of Fog': 'Visibility Issues',
    'Smoke': 'Visibility Issues',
    'Haze / Windy': 'Visibility Issues',

    'Light Snow': 'Snow/Ice',
    'Snow': 'Snow/Ice',
    'Light Snow and Sleet / Windy': 'Snow/Ice',
    'Wintry Mix': 'Snow/Ice',
    'Wintry Mix / Windy': 'Snow/Ice',
    'Light Sleet': 'Snow/Ice',
    'Heavy Sleet': 'Snow/Ice',
    'Snow and Sleet / Windy': 'Snow/Ice',
    'Heavy Snow': 'Snow/Ice',
    'Light Ice Pellets': 'Snow/Ice',
    'Light Freezing Rain': 'Snow/Ice',
    'Freezing Rain / Windy': 'Snow/Ice',

    'Widespread Dust': 'Dust/Windy',
    'Blowing Dust': 'Dust/Windy',
    'Blowing Dust / Windy': 'Dust/Windy',
    'Duststorm': 'Dust/Windy',
    
}

d2["Weather_Condition_Grouped"] = d2["Weather_Condition"].map(weather_conditions_map).fillna("Other")

In [12]:
conditions = pd.get_dummies(d2["Weather_Condition_Grouped"], prefix="Weather")
d2 = pd.concat([d2, conditions], axis=1)

In [13]:
# Converting Sunrise_Sunset to a binary variable for day and night
day_map = {
    'Night': 0,
    'Day': 1
}

d2['Day/Night'] = d2['Sunrise_Sunset'].map(day_map)

### Zone ID Creation

In [14]:
import h3

# Uses Uber's geospatial indexing library to convert lat/lng to hexagonal zones for better grouping
def get_hex_id(row):
    lat = row['Start_Lat']
    lng = row['Start_Lng']
    return h3.latlng_to_cell(lat, lng, res=6)
    

# Create the Zone_ID to plot the longitude and latitude into hexagonal zones
d2['Zone_ID'] = d2.apply(get_hex_id, axis=1)

# Making the zone into a simple integer
d2['Zone_Int_ID'], _ = pd.factorize(d2['Zone_ID'])

In [15]:
import plotly.express as px
import h3

# Aggregate accidents by the different zones in Houston - change to different cities for diff results
houston_hex = d2[d2['City_Los Angeles'] == 1].groupby('Zone_ID').size().reset_index(name='Accident_Count')

# Using GeoJSON to plot the hexagonal zones on the map with Plotly
features = []
for zone in houston_hex['Zone_ID']:
    boundary = h3.cell_to_boundary(zone)
    coords = [[lon, lat] for lat, lon in boundary]
    coords.append(coords[0])
    features.append({
        'type': 'Feature',
        'id': zone,
        'properties': {},
        'geometry': {'type': 'Polygon', 'coordinates': [coords]}
    })

geojson = {'type': 'FeatureCollection', 'features': features}

# Plotting the hexagonal zones with accident counts 
fig = px.choropleth_map(
    houston_hex,
    geojson=geojson,
    locations='Zone_ID',
    featureidkey='id',
    color='Accident_Count',
    color_continuous_scale='Reds',
    map_style='carto-positron',
    zoom=10,
    #center={'lat': 29.7604, 'lon': -95.3698}, # Coordinates for Houston
    #center={'lat': 25.7617, 'lon': -80.1918}, # Coordinates for Miami
    center={'lat': 34.0522, 'lon': -118.2437}, # Coordinates for Los Angeles
    labels={'Accident_Count': 'Accidents'}
)

fig.update_layout()
fig.show()

## Creating the Aggregated Dataset

In [ ]:
# Year col for grouping by year
d2['Year'] = d2['Start_Time'].dt.year

poi_cols = ['Amenity', 'Crossing', 'Give_Way', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal']

# Using mean to create a probability of above events occuring in a given year and zone
zone_year_profile = d2.groupby(['Zone_Int_ID', 'Year'])[poi_cols].mean().reset_index()
city_flags = d2.groupby('Zone_Int_ID')[['City_Houston', 'City_Los Angeles', 'City_Miami']].max().reset_index() 
zone_year_profile = zone_year_profile.merge(city_flags, on='Zone_Int_ID', how='left')

# Making sure that each zone has a record for each year, even if there were no accidents in that year 
years = sorted(d2['Year'].unique())
zones = zone_year_profile['Zone_Int_ID'].unique()
all_combinations = pd.MultiIndex.from_product([zones, years], names=['Zone_Int_ID', 'Year']).to_frame(index=False) 

# Merge our sparse data into this full skeleton
full_zone_profile = all_combinations.merge(zone_year_profile, on=['Zone_Int_ID', 'Year'], how='left')

# Forward filling and backward filling the missing values for these columns
full_zone_profile = full_zone_profile.sort_values(['Zone_Int_ID', 'Year'])
full_zone_profile[poi_cols] = full_zone_profile.groupby('Zone_Int_ID')[poi_cols].ffill().bfill()
full_zone_profile[['City_Houston', 'City_Los Angeles', 'City_Miami']] = full_zone_profile.groupby('Zone_Int_ID')[['City_Houston', 'City_Los Angeles', 'City_Miami']].ffill().bfill()

full_zone_profile

/var/folders/x7/sl4zn7qx64s6wq122_6jw9_00000gn/T/ipykernel_99917/2648015797.py:22: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



,Zone_Int_ID,Year,Amenity,Crossing,Give_Way,Junction,Railway,Station,Stop,Traffic_Signal,City_Houston,City_Los Angeles,City_Miami
0,0,2016,0.041169,0.233068,0.004648,0.068393,0.035857,0.037185,0.102258,0.389110,True,False,False
1,0,2017,0.039352,0.252646,0.005291,0.046296,0.049934,0.052579,0.120040,0.427249,True,False,False
2,0,2018,0.048088,0.268033,0.007095,0.035081,0.057548,0.065037,0.125345,0.458415,True,False,False
3,0,2019,0.037234,0.287643,0.010638,0.058101,0.059329,0.052373,0.129705,0.450082,True,False,False
4,0,2020,0.035184,0.236943,0.014843,0.080814,0.051127,0.053876,0.121495,0.382078,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,185,2019,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,True
1484,185,2020,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,True
1485,185,2021,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,True
1486,185,2022,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,True


In [19]:
# Making 2h interval time slowts
d2['Time_Stamp'] = d2['Start_Time'].dt.floor('2H')
min_time = d2['Time_Stamp'].min()
max_time = d2['Time_Stamp'].max()
all_times = pd.date_range(start=min_time, end=max_time, freq='2h')

# Create a DataFrame with all the time slots, so that we can merge after and still have times with no accidents
time_df = pd.DataFrame({'Time_Stamp': all_times})

/var/folders/x7/sl4zn7qx64s6wq122_6jw9_00000gn/T/ipykernel_99917/1390989738.py:2: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



In [20]:
time_df['Year'] = time_df['Time_Stamp'].dt.year
time_df['Hour'] = time_df['Time_Stamp'].dt.hour
time_df['Day_of_Week'] = time_df['Time_Stamp'].dt.dayofweek
time_df['Month'] = time_df['Time_Stamp'].dt.month
time_df['Weekend'] = (time_df['Day_of_Week'] >= 5).astype(int)

time_df['Holiday'] = time_df['Time_Stamp'].dt.date.apply(lambda x: int(x in us_holidays))

In [22]:
# Merging the time data with the zone profile data to create a value for each row
master_grid = time_df.merge(full_zone_profile, on='Year', how='inner')

In [ ]:
weather_metrics = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']
weather_ohe = ['Weather_Clear', 'Weather_Cloudy', 'Weather_Dust/Windy', 'Weather_Rain/Drizzle', 'Weather_Snow/Ice', 'Weather_Stormy', 'Weather_Visibility Issues']

all_weather = weather_metrics + weather_ohe

# Getting the mean values for all weather cols across the whole city - gives real value for metrics and pribability for the one hot encoded conditions 
city_weather_ref = d2.groupby(['City', 'Time_Stamp'])[all_weather].mean().reset_index()

master_grid['City'] = master_grid[['City_Houston', 'City_Los Angeles', 'City_Miami']].idxmax(axis=1).str.replace('City_', '')

master_grid = (master_grid.merge(city_weather_ref, on=['City', 'Time_Stamp'], how='left').sort_values(['City', 'Time_Stamp']))

# 4. Fill gaps and downcast to float32 immediately to save RAM
master_grid[all_weather] = (master_grid.groupby('City')[all_weather] .ffill().bfill().astype(float))



In [31]:
# Finding the number of accidents in each zone and time slot to create the target variable for our model
accident_counts = d2.groupby(['Zone_Int_ID', 'Time_Stamp']).size().reset_index(name='Accident_Count')

master_grid = master_grid.merge(accident_counts, on=['Zone_Int_ID', 'Time_Stamp'], how='left')

# If no accident was found in that 2-hour window, the count is zero
master_grid['Accident_Count'] = master_grid['Accident_Count'].fillna(0).astype(int)


In [39]:
modelling_dataset = master_grid.drop(columns=["City"]).copy()

modelling_dataset

,Time_Stamp,Year,Hour,Day_of_Week,Month,Weekend,Holiday,Zone_Int_ID,Amenity,Crossing,Give_Way,Junction,Railway,Station,Stop,Traffic_Signal,City_Houston,City_Los Angeles,City_Miami,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in),Weather_Clear,Weather_Cloudy,Weather_Dust/Windy,Weather_Rain/Drizzle,Weather_Snow/Ice,Weather_Stormy,Weather_Visibility Issues,Accident_Count
0,2016-03-22 18:00:00,2016,18,1,3,0,0,0,0.041169,0.233068,0.004648,0.068393,0.035857,0.037185,0.102258,0.389110,True,False,False,85.099998,68.0,29.84,8.0,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,2016-03-22 18:00:00,2016,18,1,3,0,0,1,0.030181,0.424547,0.026157,0.104628,0.014085,0.000000,0.261569,0.438632,True,False,False,85.099998,68.0,29.84,8.0,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,2016-03-22 18:00:00,2016,18,1,3,0,0,2,0.000000,0.316667,0.095833,0.058333,0.000000,0.000000,0.050000,0.350000,True,False,False,85.099998,68.0,29.84,8.0,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,2016-03-22 18:00:00,2016,18,1,3,0,0,3,0.000000,0.161290,0.000000,0.053763,0.000000,0.000000,0.021505,0.268817,True,False,False,85.099998,68.0,29.84,8.0,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,2016-03-22 18:00:00,2016,18,1,3,0,0,4,0.000000,0.021277,0.007092,0.035461,0.000000,0.000000,0.000000,0.148936,True,False,False,85.099998,68.0,29.84,8.0,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5725447,2023-03-31 20:00:00,2023,20,4,3,0,0,181,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,True,73.000000,90.0,30.09,10.0,5.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0
5725448,2023-03-31 20:00:00,2023,20,4,3,0,0,182,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,True,73.000000,90.0,30.09,10.0,5.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0
5725449,2023-03-31 20:00:00,2023,20,4,3,0,0,183,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,True,73.000000,90.0,30.09,10.0,5.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0
5725450,2023-03-31 20:00:00,2023,20,4,3,0,0,184,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,True,73.000000,90.0,30.09,10.0,5.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
# # 2 hour interval buckets
# d2['Time_Stamp'] = d2['Start_Time'].dt.floor('2h')
# # Extract hour directly from Time_Stamp so it matches the 2h bucket
# d2['Hour'] = d2['Time_Stamp'].dt.hour

# # Aggregate accidents into zone x 2hr time slot buckets (and only keeping necessary coloumns for modelling)
# # mean: weather columns (avg across accidents in bucket)
# # max: POI/weather flags (1 if any accident in bucket had it)
# # first: categorical columns (same value for all accidents in bucket)
# modelling_df = d2.groupby(['Zone_Int_ID', 'Time_Stamp']).agg(**{
#     'Mean_Severity'        : ('Severity', 'mean'),
#     'Mean_Distance(mi)'    : ('Distance(mi)', 'mean'),
#     'Mean_Temp(F)'         : ('Temperature(F)', 'mean'),
#     'Mean_Humidity(%)'     : ('Humidity(%)', 'mean'),
#     'Mean_Pressure(in)'    : ('Pressure(in)', 'mean'),
#     'Mean_Visibility(mi)'  : ('Visibility(mi)', 'mean'),
#     'Mean_Wind(mph)'       : ('Wind_Speed(mph)', 'mean'),
#     'Mean_Precip(in)'      : ('Precipitation(in)', 'mean'),
#     'Amenity'              : ('Amenity', 'max'),
#     'Crossing'             : ('Crossing', 'max'),
#     'Give_Way'             : ('Give_Way', 'max'),
#     'Junction'             : ('Junction', 'max'),
#     'Railway'              : ('Railway', 'max'),
#     'Station'              : ('Station', 'max'),
#     'Stop'                 : ('Stop', 'max'),
#     'Traffic_Signal'       : ('Traffic_Signal', 'max'),
#     'Hour'                 : ('Hour', 'first'),
#     'Day_of_Week'          : ('Day_of_Week', 'first'),
#     'Month'                : ('Month', 'first'),
#     'Weekend'              : ('Weekend', 'first'),
#     'Holiday'              : ('Holiday', 'first'),
#     'City_Houston'         : ('City_Houston', 'first'),
#     'City_LA'              : ('City_Los Angeles', 'first'),
#     'City_Miami'           : ('City_Miami', 'first'),
#     'Weather_Clear'        : ('Weather_Clear', 'max'),
#     'Weather_Cloudy'       : ('Weather_Cloudy', 'max'),
#     'Weather_Dust/Windy'   : ('Weather_Dust/Windy', 'max'),
#     'Weather_Rain/Drizzle' : ('Weather_Rain/Drizzle', 'max'),
#     'Weather_Snow/Ice'     : ('Weather_Snow/Ice', 'max'),
#     'Weather_Stormy'       : ('Weather_Stormy', 'max'),
#     'Weather_Visibility'   : ('Weather_Visibility Issues', 'max'),
#     'Day_Night'            : ('Day/Night', 'first'),
# }).reset_index()

# # Add Accident_Count target variable
# accident_counts = d2.groupby(['Zone_Int_ID', 'Time_Stamp']).size().reset_index(name='Accident_Count')
# modelling_df = modelling_df.merge(accident_counts, on=['Zone_Int_ID', 'Time_Stamp'], how='left')

In [ ]:
# modelling_df.head()

,Zone_Int_ID,Time_Stamp,Mean_Severity,Mean_Distance(mi),Mean_Temp(F),Mean_Humidity(%),Mean_Pressure(in),Mean_Visibility(mi),Mean_Wind(mph),Mean_Precip(in),Amenity,Crossing,Give_Way,Junction,Railway,Station,Stop,Traffic_Signal,Hour,Day_of_Week,Month,Weekend,Holiday,City_Houston,City_LA,City_Miami,Weather_Clear,Weather_Cloudy,Weather_Dust/Windy,Weather_Rain/Drizzle,Weather_Snow/Ice,Weather_Stormy,Weather_Visibility,Day_Night,Accident_Count
0,0,2016-06-14 20:00:00,2.0,0.000,86.0,66.0,29.840,8.0,9.2,0.0,False,False,False,False,False,True,False,True,20,1,6,False,0,True,False,False,True,False,False,False,False,False,False,1,1
1,0,2016-06-21 08:00:00,2.5,0.000,86.9,64.0,30.115,10.0,3.5,0.0,False,True,False,False,False,False,False,True,9,1,6,False,0,True,False,False,False,True,False,False,False,False,False,1,2
2,0,2016-06-21 10:00:00,2.0,0.246,87.8,58.0,30.110,9.0,4.6,0.0,False,False,False,False,False,False,True,False,11,1,6,False,0,True,False,False,False,True,False,False,False,False,False,1,1
3,0,2016-06-21 14:00:00,2.0,0.000,91.4,52.0,30.070,10.0,5.8,0.0,False,True,False,False,False,False,False,True,14,1,6,False,0,True,False,False,False,True,False,False,False,False,False,1,1
4,0,2016-06-21 18:00:00,2.0,0.000,82.4,79.0,30.020,10.0,9.2,0.0,False,False,False,False,False,False,False,True,18,1,6,False,0,True,False,False,True,False,False,False,False,False,False,1,1


In [ ]:
modelling_dataset.to_csv("modelling_dataset.csv", index=False)